<a href="https://colab.research.google.com/github/samuelamankwaa123/Textile_Model/blob/main/Cnn_textile_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet18_Weights

# Load pretrained model
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

# Modify final layer for 2 classes
model.fc = nn.Linear(model.fc.in_features, 2)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 178MB/s]


In [4]:
!apt-get install p7zip-full -y
!7z x Defect_images.7z
!7z x NODefect_images.7z

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 77676758 bytes (75 MiB)

Extracting archive: Defect_images.7z
--
Path = Defect_images.7z
Type = 7z
Physical Size = 77676758
Headers Size = 1620
Method = LZMA2:24
Solid = +
Blocks = 1

  0%     76% 80 - Defect_images/0083_029_04.png                                       Everything is Ok

Folders: 1
Files: 106
Size:       77674496
Compressed: 77676758

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p

In [5]:
import os
import shutil
from pathlib import Path

base = Path("data")
(base / "suitable").mkdir(parents=True, exist_ok=True)
(base / "unsuitable").mkdir(parents=True, exist_ok=True)

# copy no-defect images to suitable
src_good = Path("NODefect_images")
for f in src_good.rglob("*"):
    if f.is_file():
        shutil.copy(f, base / "suitable" / f.name)

# copy defect images to unsuitable
src_bad = Path("Defect_images")
for f in src_bad.rglob("*"):
    if f.is_file():
        shutil.copy(f, base / "unsuitable" / f.name)

print("Suitable:", len(list((base / "suitable").glob("*"))))
print("Unsuitable:", len(list((base / "unsuitable").glob("*"))))

Suitable: 141
Unsuitable: 106


In [6]:
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader
import torch

# Use the normalization expected by pretrained ResNet18
weights = ResNet18_Weights.DEFAULT
mean = weights.transforms().mean
std = weights.transforms().std

# Training transforms: add augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Validation transforms: no augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Load once without transforms so we can make a stable split
full_dataset = datasets.ImageFolder("data")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(full_dataset), generator=generator).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

# Reload with different transforms
train_dataset = datasets.ImageFolder("data", transform=train_transform)
val_dataset = datasets.ImageFolder("data", transform=val_transform)

# Apply split indices
train_dataset.samples = [train_dataset.samples[i] for i in train_indices]
train_dataset.targets = [train_dataset.targets[i] for i in train_indices]

val_dataset.samples = [val_dataset.samples[i] for i in val_indices]
val_dataset.targets = [val_dataset.targets[i] for i in val_indices]

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("Classes:", train_dataset.classes)
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Classes: ['suitable', 'unsuitable']
Train: 197
Val: 50


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Move model to device ---
model = model.to(device)

# --- Unfreeze last ResNet block (important improvement) ---
for param in model.layer4.parameters():
    param.requires_grad = True

# --- Loss ---
criterion = nn.CrossEntropyLoss()

# --- Optimizer (different LR for layers) ---
optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-4},
])

# --- Scheduler (reduces LR over time) ---
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

epochs = 10

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    # -------- TRAIN --------
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # -------- VALIDATION --------
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc = correct / total

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.3f}")

    # -------- SAVE BEST MODEL --------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, "best_model.pth")
        print("✅ Saved best model")

print(f"\nBest Validation Accuracy: {best_val_acc:.3f}")


Epoch 1/10
Train Loss: 0.7021 | Train Acc: 0.584
Val   Loss: 0.6085 | Val   Acc: 0.660
✅ Saved best model

Epoch 2/10
Train Loss: 0.6621 | Train Acc: 0.604
Val   Loss: 0.5421 | Val   Acc: 0.740
✅ Saved best model

Epoch 3/10
Train Loss: 0.6242 | Train Acc: 0.660
Val   Loss: 0.5511 | Val   Acc: 0.740

Epoch 4/10
Train Loss: 0.6171 | Train Acc: 0.670
Val   Loss: 0.5465 | Val   Acc: 0.760
✅ Saved best model

Epoch 5/10
Train Loss: 0.5656 | Train Acc: 0.746
Val   Loss: 0.5296 | Val   Acc: 0.780
✅ Saved best model

Epoch 6/10
Train Loss: 0.5855 | Train Acc: 0.680
Val   Loss: 0.5386 | Val   Acc: 0.780

Epoch 7/10
Train Loss: 0.5659 | Train Acc: 0.706
Val   Loss: 0.5323 | Val   Acc: 0.760

Epoch 8/10
Train Loss: 0.5557 | Train Acc: 0.711
Val   Loss: 0.5233 | Val   Acc: 0.720

Epoch 9/10
Train Loss: 0.5505 | Train Acc: 0.751
Val   Loss: 0.5226 | Val   Acc: 0.740

Epoch 10/10
Train Loss: 0.5362 | Train Acc: 0.711
Val   Loss: 0.5288 | Val   Acc: 0.720

Best Validation Accuracy: 0.780
